In [ ]:
# Paste and run this in a Colab cell (replace NOTEBOOK path if needed) - CORRECT

# 0. (first time) installs + ffmpeg + pyngrok
!pip install -q flask flask-cors pyngrok
!apt-get -qq update && apt-get -qq install -y ffmpeg
# If needed:
!ngrok authtoken 2wVOhcrhm9F6C9TUzmHet24C6Sf_6Mh2EmMTQ5jVUQx77tWkH

# 1. Imports & dirs
import os, subprocess, threading, traceback, time
from flask import Flask, request, jsonify, send_from_directory, send_file, make_response, abort
from flask_cors import CORS
from pyngrok import ngrok

os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

# 2. Constants (adjust NOTEBOOK path to your notebook file name)
INPUT_PATH = '/content/input/test.mp4'
OUT_VIDEO  = '/content/summary_video.mp4'           # produced by your notebook
OUT_FIXED  = '/content/output/summary_fixed.mp4'     # re-encoded for browser
OUT_TEXT   = '/content/output/summary.txt'           # textual summary expected
MODE_PATH  = '/content/input/mode.txt'
NOTEBOOK   = '/content/Full_Video_Summarization_final.ipynb'  # <-- change if needed

# 3. Flask app + CORS
app = Flask(__name__)
CORS(app)  # allow cross-origin from frontend

# 4. Re-encode function (makes MP4 web-friendly)
def reencode_video_for_web(input_path, output_path):
    try:
        print("🔁 Re-encoding", input_path, "->", output_path)
        subprocess.run([
            'ffmpeg', '-y', '-i', input_path,
            '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
            '-c:a', 'aac', '-b:a', '128k',
            '-movflags', '+faststart',
            output_path
        ], check=True)
        print("✅ Re-encoding complete.")
    except Exception as e:
        print("❌ FFmpeg re-encoding error:", e)

# 5. Background notebook runner
def run_notebook():
    print("🔁 Starting notebook execution (background)...")
    try:
        # read mode if exists and export as env var so notebook cells can read os.environ.get('MODE')
        mode = None
        if os.path.exists(MODE_PATH):
            try:
                with open(MODE_PATH, 'r') as fh:
                    mode = fh.read().strip()
                print("ℹ️ Notebook mode:", mode)
            except:
                pass

        env = os.environ.copy()
        if mode:
            env['MODE'] = mode

        # Execute the notebook (nbconvert executes the notebook top-to-bottom)
        subprocess.run([
            'jupyter', 'nbconvert', '--to', 'notebook', '--execute', NOTEBOOK,
            '--output', 'executed.ipynb',
            '--ExecutePreprocessor.timeout=1800'  # 30 min - increase if your runs take longer
        ], check=True, env=env)
        print("✅ Notebook execution finished.")

        # After notebook completes, re-encode OUT_VIDEO to OUT_FIXED for browser compatibility
        if os.path.exists(OUT_VIDEO):
            reencode_video_for_web(OUT_VIDEO, OUT_FIXED)
        else:
            print("⚠️ Notebook finished but OUT_VIDEO not found:", OUT_VIDEO)

    except Exception:
        print("❌ Error during notebook execution:\n", traceback.format_exc())

# 6. /upload endpoint - receives form-data 'video' and optional 'mode'
@app.route('/upload', methods=['POST'])
def upload_video():
    try:
        f = request.files.get('video')
        mode = request.form.get('mode', '').strip()

        if not f:
            return "No file received (field 'video')", 400

        # Save file as fixed path (test.mp4)
        f.save(INPUT_PATH)
        print(f"📥 Saved uploaded file to {INPUT_PATH}")

        # Save the mode to a small text file, for notebook to read
        if mode:
            with open(MODE_PATH, 'w') as fh:
                fh.write(mode)
            print("📌 Saved mode:", mode)
        else:
            if os.path.exists(MODE_PATH):
                os.remove(MODE_PATH)
                print("🧹 Removed previous mode file")

        # Remove old outputs to avoid confusion
        for p in (OUT_VIDEO, OUT_FIXED, OUT_TEXT):
            if os.path.exists(p):
                os.remove(p)
                print("🧹 Removed old file:", p)

        # Start notebook execution in background thread (non-blocking)
        t = threading.Thread(target=run_notebook, daemon=True)
        t.start()

        return "Processing started", 200

    except Exception as e:
        print("❌ Upload error:", e)
        return str(e), 500

# 7. /check endpoint - the frontend polls this to know when result is ready
@app.route('/check', methods=['GET'])
def check_progress():
    # check the *re-encoded* fixed path for video readiness
    video_ready = os.path.exists(OUT_FIXED)
    text_ready  = os.path.exists(OUT_TEXT)
    print(f"[check] video_ready={video_ready} text_ready={text_ready} time={time.ctime()}")
    if video_ready and text_ready:
        try:
            with open(OUT_TEXT, 'r') as fh:
                summary = fh.read()
        except:
            summary = ''
        return jsonify(ready=True, summary=summary)
    return jsonify(ready=False)

# 8. Serve raw summary text at /summary.txt with CORS-friendly headers
@app.route('/summary.txt')
def serve_summary_txt():
    if not os.path.exists(OUT_TEXT):
        return ("Not ready", 404)
    try:
        with open(OUT_TEXT, 'r') as fh:
            txt = fh.read()
        resp = make_response(txt)
        resp.headers['Content-Type'] = 'text/plain; charset=utf-8'
        resp.headers['Access-Control-Allow-Origin'] = '*'
        # disable caching so frontend always fetches fresh file
        resp.headers['Cache-Control'] = 'no-cache, no-store, must-revalidate'
        return resp
    except Exception as e:
        print("❌ serve_summary_txt error:", e)
        return ("Server error", 500)

# 9. Serve re-encoded summary directly at /summary.mp4
@app.route('/summary.mp4')
def serve_summary_video():
    if not os.path.exists(OUT_FIXED):
        return ("Not ready", 404)
    try:
        resp = make_response(send_file(OUT_FIXED, mimetype='video/mp4'))
        resp.headers['Access-Control-Allow-Origin'] = '*'
        # tell browser not to cache (for polling)
        resp.headers['Cache-Control'] = 'no-cache, no-store, must-revalidate'
        return resp
    except Exception as e:
        print("❌ serve_summary_video error:", e)
        return ("Server error", 500)

# 10. Serve output folder generically (optional)
@app.route('/output/<path:filename>')
def serve_output(filename):
    return send_from_directory('/content/output', filename)

# 11. Start ngrok + Flask (run last)
if __name__ == '__main__':
    public_url = ngrok.connect(5000).public_url
    print("🚀 PUBLIC URL:", public_url)
    print("Set BACKEND_URL in your frontend to this value (no trailing slash).")
    app.run(port=5000, host='0.0.0.0')


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🚀 PUBLIC URL: https://eec48cd9710f.ngrok-free.app
Set BACKEND_URL in your frontend to this value (no trailing slash).
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:35:37] "POST /upload HTTP/1.1" 200 -


📥 Saved uploaded file to /content/input/test.mp4
📌 Saved mode: full
🔁 Starting notebook execution (background)...
ℹ️ Notebook mode: full


INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:35:48] "HEAD /output/summary_fixed.mp4?nocache=1758098147713 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:35:58] "HEAD /output/summary_fixed.mp4?nocache=1758098157972 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:36:08] "HEAD /output/summary_fixed.mp4?nocache=1758098168194 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:36:18] "HEAD /output/summary_fixed.mp4?nocache=1758098177979 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:36:28] "HEAD /output/summary_fixed.mp4?nocache=1758098187969 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:36:38] "HEAD /output/summary_fixed.mp4?nocache=1758098197985 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:36:48] "HEAD /output/summary_fixed.mp4?nocache=1758098207988 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:37:01] "HEAD /output/summary_fixed.mp4?nocache=1758098220980 HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [17/

✅ Notebook execution finished.
🔁 Re-encoding /content/summary_video.mp4 -> /content/output/summary_fixed.mp4


INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:43:08] "HEAD /output/summary_fixed.mp4?nocache=1758098587973 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:43:38] "GET /output/summary_fixed.mp4?nocache=1758098588789 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:43:40] "GET /output/summary_fixed.mp4?nocache=1758098588789 HTTP/1.1" 206 -


✅ Re-encoding complete.


INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:43:56] "GET /output/summary_fixed.mp4?nocache=1758098588789 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:43:57] "GET /output/summary_fixed.mp4?nocache=1758098588789 HTTP/1.1" 206 -
INFO:werkzeug:127.0.0.1 - - [17/Sep/2025 08:43:58] "GET /output/summary_fixed.mp4?nocache=1758098588789 HTTP/1.1" 206 -
